# HOL 1: Data Engineering Pipeline
## Raw → Silver → Gold Transformation with SQL

In this hands-on lab, you will build a data transformation pipeline using SQL.

**What you'll learn:**
- Building a medallion architecture (Raw → Silver → Gold)
- Data cleansing: deduplication, filtering, enrichment
- Data quality profiling techniques
- Creating business-ready aggregated tables

**Prerequisites:** You must have run `setup.sql` first.

---

## Step 1: Set Context and Verify Data

Run the cell below to set your session context and verify the raw data is available.

In [ ]:

USE ROLE ACCOUNTADMIN;
USE DATABASE BB_TRAINING;
USE SCHEMA RAW;
USE WAREHOUSE BB_TRAINING_WH;

In [ ]:

-- Verify raw data exists
SELECT 'RAW_CUSTOMERS' AS table_name, COUNT(*) AS row_count FROM RAW.RAW_CUSTOMERS
UNION ALL SELECT 'RAW_LOAN_APPLICATIONS', COUNT(*) FROM RAW.RAW_LOAN_APPLICATIONS
UNION ALL SELECT 'RAW_TRANSACTIONS', COUNT(*) FROM RAW.RAW_TRANSACTIONS
UNION ALL SELECT 'RAW_RISK_ASSESSMENTS', COUNT(*) FROM RAW.RAW_RISK_ASSESSMENTS
ORDER BY table_name;

---
## Step 2: Silver Layer - Clean & Validate

The Silver layer applies:
- **Deduplication** - Remove any duplicate records
- **Filtering** - Remove invalid or irrelevant rows
- **Standardization** - Consistent formatting
- **Enrichment** - Add calculated fields

We'll create 3 Silver tables:
1. `SILVER_CUSTOMERS` - Clean customer profiles
2. `SILVER_LOAN_APPLICATIONS` - Enriched with risk data
3. `SILVER_TRANSACTIONS` - Validated payment records

### 2a. SILVER_CUSTOMERS
Clean and standardize customer profiles:
- Deduplicate on `CUSTOMER_ID` (keep most recent)
- Filter out null/invalid ABNs
- Add `REVENUE_BAND` and `EMPLOYEE_BAND` classifications

In [ ]:

CREATE OR REPLACE TABLE SILVER.SILVER_CUSTOMERS AS
WITH deduplicated AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY CUSTOMER_ID ORDER BY CREATED_AT DESC) AS row_num
    FROM RAW.RAW_CUSTOMERS
    WHERE ABN IS NOT NULL
      AND LENGTH(ABN) = 11
)
SELECT
    CUSTOMER_ID,
    BUSINESS_NAME,
    ABN,
    INDUSTRY,
    UPPER(TRIM(STATE)) AS STATE,
    ANNUAL_REVENUE,
    EMPLOYEES,
    YEARS_IN_BUSINESS,
    REGISTRATION_DATE,
    CREATED_AT,
    -- Revenue band classification
    CASE
        WHEN ANNUAL_REVENUE < 500000 THEN 'Micro'
        WHEN ANNUAL_REVENUE < 2000000 THEN 'Small'
        WHEN ANNUAL_REVENUE < 10000000 THEN 'Medium'
        ELSE 'Large'
    END AS REVENUE_BAND,
    -- Employee band classification
    CASE
        WHEN EMPLOYEES < 5 THEN '1-4'
        WHEN EMPLOYEES < 20 THEN '5-19'
        WHEN EMPLOYEES < 100 THEN '20-99'
        ELSE '100+'
    END AS EMPLOYEE_BAND
FROM deduplicated
WHERE row_num = 1;

In [ ]:

-- Verify
SELECT COUNT(*) AS silver_customers_count FROM SILVER.SILVER_CUSTOMERS;

### 2b. SILVER_LOAN_APPLICATIONS
Enrich loan applications with risk assessment data:
- Join with risk assessments (1:1 on `APPLICATION_ID`)
- Filter out Withdrawn applications
- Add `DAYS_TO_DECISION`, `APPLICATION_MONTH`, and `IS_APPROVED` fields

In [ ]:

CREATE OR REPLACE TABLE SILVER.SILVER_LOAN_APPLICATIONS AS
SELECT
    a.APPLICATION_ID,
    a.CUSTOMER_ID,
    a.APPLICATION_DATE,
    a.LOAN_AMOUNT,
    a.LOAN_PRODUCT,
    a.LOAN_TERM_MONTHS,
    a.INTEREST_RATE,
    a.PURPOSE,
    a.STATUS,
    a.DECISION_DATE,
    a.CREATED_AT,
    -- Enrichment from risk assessments
    r.CREDIT_SCORE,
    r.RISK_TIER,
    r.DEBT_SERVICE_RATIO,
    r.COLLATERAL_VALUE,
    r.MODEL_VERSION,
    -- Calculated fields
    DATEDIFF('day', a.APPLICATION_DATE, a.DECISION_DATE) AS DAYS_TO_DECISION,
    DATE_TRUNC('month', a.APPLICATION_DATE) AS APPLICATION_MONTH,
    CASE WHEN a.STATUS = 'Approved' THEN 1 ELSE 0 END AS IS_APPROVED
FROM RAW.RAW_LOAN_APPLICATIONS a
LEFT JOIN RAW.RAW_RISK_ASSESSMENTS r
    ON a.APPLICATION_ID = r.APPLICATION_ID
WHERE a.STATUS != 'Withdrawn';

In [ ]:

-- Verify
SELECT COUNT(*) AS silver_applications_count FROM SILVER.SILVER_LOAN_APPLICATIONS;

### 2c. SILVER_TRANSACTIONS
Validate and clean transaction records:
- Remove Failed transactions
- Ensure positive amounts
- Add `IS_LATE_PAYMENT`, `TRANSACTION_MONTH`, and `AMOUNT_CATEGORY` fields

In [ ]:

CREATE OR REPLACE TABLE SILVER.SILVER_TRANSACTIONS AS
SELECT
    TRANSACTION_ID,
    LOAN_ID,
    CUSTOMER_ID,
    TRANSACTION_DATE,
    AMOUNT,
    TRANSACTION_TYPE,
    STATUS,
    PAYMENT_METHOD,
    CREATED_AT,
    -- Calculated fields
    CASE WHEN TRANSACTION_TYPE = 'Late Fee' THEN TRUE ELSE FALSE END AS IS_LATE_PAYMENT,
    DATE_TRUNC('month', TRANSACTION_DATE) AS TRANSACTION_MONTH,
    CASE
        WHEN AMOUNT < 1000 THEN 'Small'
        WHEN AMOUNT < 10000 THEN 'Medium'
        WHEN AMOUNT < 100000 THEN 'Large'
        ELSE 'Very Large'
    END AS AMOUNT_CATEGORY
FROM RAW.RAW_TRANSACTIONS
WHERE STATUS != 'Failed'
  AND AMOUNT > 0;

In [ ]:

-- Verify
SELECT COUNT(*) AS silver_transactions_count FROM SILVER.SILVER_TRANSACTIONS;

---
## Step 3: Data Quality Profiling

Before building the Gold layer, let's profile our Silver data to understand:
- How much data was filtered (raw vs silver counts)
- Distribution of key dimensions
- This is a critical step in any data engineering pipeline!

In [ ]:

-- Data Quality: Row count comparison (Raw vs Silver)
SELECT 'Customers' AS table_name,
       (SELECT COUNT(*) FROM RAW.RAW_CUSTOMERS) AS raw_count,
       (SELECT COUNT(*) FROM SILVER.SILVER_CUSTOMERS) AS silver_count,
       (SELECT COUNT(*) FROM RAW.RAW_CUSTOMERS) - (SELECT COUNT(*) FROM SILVER.SILVER_CUSTOMERS) AS filtered
UNION ALL
SELECT 'Loan Applications',
       (SELECT COUNT(*) FROM RAW.RAW_LOAN_APPLICATIONS),
       (SELECT COUNT(*) FROM SILVER.SILVER_LOAN_APPLICATIONS),
       (SELECT COUNT(*) FROM RAW.RAW_LOAN_APPLICATIONS) - (SELECT COUNT(*) FROM SILVER.SILVER_LOAN_APPLICATIONS)
UNION ALL
SELECT 'Transactions',
       (SELECT COUNT(*) FROM RAW.RAW_TRANSACTIONS),
       (SELECT COUNT(*) FROM SILVER.SILVER_TRANSACTIONS),
       (SELECT COUNT(*) FROM RAW.RAW_TRANSACTIONS) - (SELECT COUNT(*) FROM SILVER.SILVER_TRANSACTIONS);

In [ ]:

-- Data Quality: Customer distribution by Industry
SELECT INDUSTRY, COUNT(*) AS customer_count
FROM SILVER.SILVER_CUSTOMERS
GROUP BY INDUSTRY
ORDER BY customer_count DESC;

In [ ]:

-- Data Quality: Application status distribution with avg loan amount
SELECT STATUS, 
       COUNT(*) AS count,
       ROUND(AVG(LOAN_AMOUNT), 2) AS avg_loan_amount,
       ROUND(AVG(CREDIT_SCORE), 0) AS avg_credit_score
FROM SILVER.SILVER_LOAN_APPLICATIONS
GROUP BY STATUS
ORDER BY count DESC;

In [ ]:

-- Data Quality: Risk tier distribution
SELECT RISK_TIER, 
       COUNT(*) AS count,
       ROUND(AVG(CREDIT_SCORE), 0) AS avg_credit_score
FROM SILVER.SILVER_LOAN_APPLICATIONS
WHERE RISK_TIER IS NOT NULL
GROUP BY RISK_TIER
ORDER BY avg_credit_score DESC;

---
## Step 4: Gold Layer - Business-Ready Aggregates

The Gold layer creates business-ready tables optimized for analytics:
1. **GOLD_LOAN_PERFORMANCE** - Per-loan metrics (repayment tracking)
2. **GOLD_CUSTOMER_360** - Complete customer view (all loans + risk)
3. **GOLD_PORTFOLIO_SUMMARY** - Monthly portfolio metrics (executive reporting)

### 4a. GOLD_LOAN_PERFORMANCE
Per-loan metrics: total repaid, late payment counts, repayment ratio, and a health score.

In [ ]:

CREATE OR REPLACE TABLE GOLD.GOLD_LOAN_PERFORMANCE AS
WITH loan_txn_summary AS (
    SELECT
        LOAN_ID,
        SUM(CASE WHEN TRANSACTION_TYPE = 'Repayment' THEN AMOUNT ELSE 0 END) AS TOTAL_REPAID,
        SUM(CASE WHEN TRANSACTION_TYPE = 'Interest' THEN AMOUNT ELSE 0 END) AS TOTAL_INTEREST_PAID,
        SUM(CASE WHEN TRANSACTION_TYPE = 'Late Fee' THEN AMOUNT ELSE 0 END) AS TOTAL_LATE_FEES,
        COUNT(CASE WHEN TRANSACTION_TYPE = 'Late Fee' THEN 1 END) AS LATE_PAYMENT_COUNT,
        COUNT(CASE WHEN TRANSACTION_TYPE = 'Repayment' THEN 1 END) AS REPAYMENT_COUNT,
        MAX(TRANSACTION_DATE) AS LAST_PAYMENT_DATE
    FROM SILVER.SILVER_TRANSACTIONS
    GROUP BY LOAN_ID
)
SELECT
    a.APPLICATION_ID,
    a.CUSTOMER_ID,
    a.LOAN_PRODUCT,
    a.LOAN_AMOUNT,
    a.LOAN_TERM_MONTHS,
    a.INTEREST_RATE,
    a.RISK_TIER,
    a.CREDIT_SCORE,
    a.APPLICATION_DATE,
    a.DECISION_DATE,
    COALESCE(t.TOTAL_REPAID, 0) AS TOTAL_REPAID,
    COALESCE(t.TOTAL_INTEREST_PAID, 0) AS TOTAL_INTEREST_PAID,
    COALESCE(t.TOTAL_LATE_FEES, 0) AS TOTAL_LATE_FEES,
    COALESCE(t.LATE_PAYMENT_COUNT, 0) AS LATE_PAYMENT_COUNT,
    COALESCE(t.REPAYMENT_COUNT, 0) AS REPAYMENT_COUNT,
    t.LAST_PAYMENT_DATE,
    -- Repayment ratio (% of loan repaid)
    ROUND(COALESCE(t.TOTAL_REPAID, 0) / a.LOAN_AMOUNT * 100, 2) AS REPAYMENT_RATIO,
    -- Payment health score (0-100, higher is better)
    GREATEST(0,
        100
        - (COALESCE(t.LATE_PAYMENT_COUNT, 0) * 10)
        - CASE WHEN COALESCE(t.TOTAL_REPAID, 0) / a.LOAN_AMOUNT * 100 < 20 THEN 20 ELSE 0 END
    ) AS PAYMENT_HEALTH_SCORE
FROM SILVER.SILVER_LOAN_APPLICATIONS a
LEFT JOIN loan_txn_summary t
    ON a.APPLICATION_ID = t.LOAN_ID
WHERE a.STATUS = 'Approved';

In [ ]:

-- Verify
SELECT COUNT(*) AS gold_loan_performance_count FROM GOLD.GOLD_LOAN_PERFORMANCE;
SELECT * FROM GOLD.GOLD_LOAN_PERFORMANCE LIMIT 5;

### 4b. GOLD_CUSTOMER_360
One row per customer with aggregated loan portfolio metrics and risk categorization.

In [ ]:

CREATE OR REPLACE TABLE GOLD.GOLD_CUSTOMER_360 AS
WITH customer_loan_metrics AS (
    SELECT
        CUSTOMER_ID,
        COUNT(*) AS TOTAL_LOANS,
        SUM(LOAN_AMOUNT) AS TOTAL_EXPOSURE,
        SUM(TOTAL_REPAID) AS TOTAL_REPAID,
        AVG(CREDIT_SCORE) AS AVG_CREDIT_SCORE,
        AVG(PAYMENT_HEALTH_SCORE) AS AVG_PAYMENT_HEALTH,
        SUM(LATE_PAYMENT_COUNT) AS TOTAL_LATE_PAYMENTS,
        MAX(LAST_PAYMENT_DATE) AS LAST_ACTIVITY_DATE,
        AVG(INTEREST_RATE) AS AVG_INTEREST_RATE
    FROM GOLD.GOLD_LOAN_PERFORMANCE
    GROUP BY CUSTOMER_ID
)
SELECT
    c.CUSTOMER_ID,
    c.BUSINESS_NAME,
    c.INDUSTRY,
    c.STATE,
    c.ANNUAL_REVENUE,
    c.EMPLOYEES,
    c.YEARS_IN_BUSINESS,
    c.REVENUE_BAND,
    c.EMPLOYEE_BAND,
    COALESCE(m.TOTAL_LOANS, 0) AS TOTAL_LOANS,
    COALESCE(m.TOTAL_EXPOSURE, 0) AS TOTAL_EXPOSURE,
    COALESCE(m.TOTAL_REPAID, 0) AS TOTAL_REPAID,
    m.AVG_CREDIT_SCORE,
    m.AVG_PAYMENT_HEALTH,
    COALESCE(m.TOTAL_LATE_PAYMENTS, 0) AS TOTAL_LATE_PAYMENTS,
    m.LAST_ACTIVITY_DATE,
    m.AVG_INTEREST_RATE,
    -- Customer risk category
    CASE
        WHEN m.AVG_PAYMENT_HEALTH >= 80 THEN 'Low Risk'
        WHEN m.AVG_PAYMENT_HEALTH >= 50 THEN 'Medium Risk'
        WHEN m.AVG_PAYMENT_HEALTH IS NOT NULL THEN 'High Risk'
        ELSE 'No Loan History'
    END AS CUSTOMER_RISK_CATEGORY
FROM SILVER.SILVER_CUSTOMERS c
LEFT JOIN customer_loan_metrics m
    ON c.CUSTOMER_ID = m.CUSTOMER_ID;

In [ ]:

-- Verify
SELECT COUNT(*) AS gold_customer_360_count FROM GOLD.GOLD_CUSTOMER_360;
SELECT * FROM GOLD.GOLD_CUSTOMER_360 LIMIT 5;

### 4c. GOLD_PORTFOLIO_SUMMARY
Monthly portfolio metrics by industry, state, and loan product for executive reporting.

In [ ]:

CREATE OR REPLACE TABLE GOLD.GOLD_PORTFOLIO_SUMMARY AS
SELECT
    a.APPLICATION_MONTH,
    c.INDUSTRY,
    c.STATE,
    a.LOAN_PRODUCT,
    COUNT(*) AS TOTAL_APPLICATIONS,
    SUM(a.IS_APPROVED) AS TOTAL_APPROVALS,
    SUM(a.LOAN_AMOUNT) AS TOTAL_LOAN_AMOUNT,
    ROUND(AVG(a.LOAN_AMOUNT), 2) AS AVG_LOAN_AMOUNT,
    ROUND(AVG(a.CREDIT_SCORE), 0) AS AVG_CREDIT_SCORE,
    ROUND(AVG(a.DAYS_TO_DECISION), 1) AS AVG_DAYS_TO_DECISION,
    COUNT(DISTINCT a.CUSTOMER_ID) AS UNIQUE_CUSTOMERS,
    ROUND(SUM(a.IS_APPROVED) / COUNT(*) * 100, 1) AS APPROVAL_RATE
FROM SILVER.SILVER_LOAN_APPLICATIONS a
LEFT JOIN SILVER.SILVER_CUSTOMERS c
    ON a.CUSTOMER_ID = c.CUSTOMER_ID
GROUP BY a.APPLICATION_MONTH, c.INDUSTRY, c.STATE, a.LOAN_PRODUCT
ORDER BY a.APPLICATION_MONTH DESC, c.INDUSTRY;

In [ ]:

-- Verify
SELECT COUNT(*) AS gold_portfolio_summary_count FROM GOLD.GOLD_PORTFOLIO_SUMMARY;
SELECT * FROM GOLD.GOLD_PORTFOLIO_SUMMARY LIMIT 10;

---
## Step 5: Final Verification

Let's verify our complete pipeline output.

In [ ]:

-- Final Pipeline Summary: All tables across all layers
SELECT 'RAW' AS layer, 'RAW_CUSTOMERS' AS table_name, COUNT(*) AS rows FROM RAW.RAW_CUSTOMERS
UNION ALL SELECT 'RAW', 'RAW_LOAN_APPLICATIONS', COUNT(*) FROM RAW.RAW_LOAN_APPLICATIONS
UNION ALL SELECT 'RAW', 'RAW_TRANSACTIONS', COUNT(*) FROM RAW.RAW_TRANSACTIONS
UNION ALL SELECT 'RAW', 'RAW_RISK_ASSESSMENTS', COUNT(*) FROM RAW.RAW_RISK_ASSESSMENTS
UNION ALL SELECT 'SILVER', 'SILVER_CUSTOMERS', COUNT(*) FROM SILVER.SILVER_CUSTOMERS
UNION ALL SELECT 'SILVER', 'SILVER_LOAN_APPLICATIONS', COUNT(*) FROM SILVER.SILVER_LOAN_APPLICATIONS
UNION ALL SELECT 'SILVER', 'SILVER_TRANSACTIONS', COUNT(*) FROM SILVER.SILVER_TRANSACTIONS
UNION ALL SELECT 'GOLD', 'GOLD_LOAN_PERFORMANCE', COUNT(*) FROM GOLD.GOLD_LOAN_PERFORMANCE
UNION ALL SELECT 'GOLD', 'GOLD_CUSTOMER_360', COUNT(*) FROM GOLD.GOLD_CUSTOMER_360
UNION ALL SELECT 'GOLD', 'GOLD_PORTFOLIO_SUMMARY', COUNT(*) FROM GOLD.GOLD_PORTFOLIO_SUMMARY
ORDER BY layer, table_name;

---
## Congratulations!

You've successfully built a complete data engineering pipeline using SQL:

- **Raw Layer** (4 tables) - Landing zone with synthetic business lending data
- **Silver Layer** (3 tables) - Cleaned, validated, and enriched
- **Gold Layer** (3 tables) - Business-ready aggregates for analytics

### Key Concepts Applied:
- `CREATE OR REPLACE TABLE ... AS SELECT` for idempotent transformations
- `ROW_NUMBER()` window function for deduplication
- `LEFT JOIN` for data enrichment
- `CASE` expressions for classification and calculated fields
- `GROUP BY` aggregations for Gold layer summaries
- Data quality profiling between pipeline layers

### Next: Build a Streamlit Dashboard
Continue to the Streamlit section to visualize your Gold layer data!

# HOL 1: Data Engineering Pipeline
## Raw → Silver → Gold Transformation with Snowpark Python

In this hands-on lab, you will build a data transformation pipeline using Snowpark Python.

**What you'll learn:**
- How to use Snowpark DataFrames to transform data
- Building a medallion architecture (Raw → Silver → Gold)
- Data quality profiling techniques
- Creating business-ready aggregated tables

**Prerequisites:** You must have run `setup.sql` first.

---